In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=gemini_api_key)

In [12]:
query = "현재 박스오피스 1위 영화는?"

In [13]:
response = model.invoke(query)
response

AIMessage(content="현재(2024년 6월 6일 오전 기준) 박스오피스 1위 영화는 **'원더랜드'** 입니다.\n\n다만, 박스오피스 순위는 매일 실시간으로 변동될 수 있으니, 가장 정확한 최신 정보는 영화진흥위원회(KOFIC) 박스오피스 또는 주요 예매 사이트(네이버 영화, 다음 영화 등)에서 확인하시는 것이 좋습니다.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--345d5d45-c683-4ca5-8e13-4157e872de50-0', usage_metadata={'input_tokens': 13, 'output_tokens': 1196, 'total_tokens': 1209, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 1095}})

In [14]:
from langchain_community.tools import DuckDuckGoSearchResults 

search = DuckDuckGoSearchResults(results_separator=';\n')
docs = search.invoke(query)

print(docs)

snippet: 지난 13일 개봉 이후 일일 박스오피스 1 위를 유지하던 리들리 스콧 감독의 신작 ‘글래디에이터 2’는 2만 8천여 명의 관객을 기록하며 3위로 내려왔습니다., title: ‘위키드’ 개봉 첫날 8만4천여명 동원, 박스오피스 1 위 …‘히든 페이스’ 2위, link: https://news.kbs.co.kr/news/pc/view/view.do?ncd=8111781;
snippet: [KtN 신미희기자] 폭우도 막지 못한 박찬욱의 신작 흡인력, 는 감독의 필모그래피를 넘어 한국영화 흥행 지형을 새롭게 써 내려가고 있다.박찬욱 감독의 12번째 장편영화 가 개봉 첫날 33만1518명의 관객을 동원하며 박스오피스 1 위에 올랐다., title: ' 박스오피스 1 위 ' 추석 황금연휴까지... - KtN (K trendy NEWS), link: https://www.k-trendynews.com/news/articleView.html?idxno=192210;
snippet: 28일 영화진흥위원회 영화관입장권 통합전산망에 따르면 'F1 더 무비'는 이날 6만 5988명의 관객을 동원하며 박스오피스 1 위를 차지했다. 지난 25일 개봉 당일부터 이 영화는 8만 2017명의 관객들을 영화관으로 이끌며 순조로운 시작을 알렸다., title: 새 판 짜는 여름 극장가, 'F1 더무비' 박스오피스 1 위 선점, link: https://v.daum.net/v/20250628102331919;
snippet: Moyens I/O » 영화 및 TV » 지금 너를 보인다: 박스오피스 1 위 , 러닝맨을 이기다.이 영화는 여전히 관객을 끌어모으고 있습니다., title: 지금 너를 보인다: 박스오피스 1 위 , 러닝맨을 이기다 | Moyens I/O, link: https://kr.moyens.net/movies-tv/386194/


### 🔎 웹 검색 도구

In [17]:
from langchain_core.tools import tool
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

@tool
def get_web_search(query: str, search_period: str='w') -> str:
    """
    웹 검색을 수행하는 함수.

    Args:
        query (str): 검색어
        search_period (str): 검색 기간 (e.g., "w" for past week (default), "m" for past month, "y" for past year, "d" for past day)

    Returns:
        str: 검색 결과
    """
    wrapper = DuckDuckGoSearchAPIWrapper(
        region="kr-kr",
        time=search_period
    )

    print('\n----- WEB SEARCH -----')
    print(query)
    print(search_period)

    search = DuckDuckGoSearchResults(
        api_wrapper=wrapper,
        source="news",
        results_separator=';\n'
    )

    searched = search.invoke(query)
    
    for i, result in enumerate(searched.split(';\n')):
        print(f'{i+1}. {result}')
    
    return searched

In [18]:
from langchain.agents import create_agent

# 도구들을 tools 리스트에 추가
tools = [get_web_search,]

# 에이전트 생성
agent = create_agent(
    model,
    tools=tools,
    system_prompt="너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."
)

In [19]:
# Run the agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)


----- WEB SEARCH -----
현재 박스오피스 1위 영화
w
1. snippet: Baukostenersparnis: Bei Bestellung eines 1 & 1 Glasfaser-Direktanschlusses bis zum Ende der Vorvermarktungsphase entfallen die Kosten für die ..., title: 1&1 Internet und Mobilfunk > Alles aus einer Hand, link: https://www.1und1.de/
2. snippet: ... Lap: Kimi Antonelli (Mercedes) - 1 :30 ... Fastest Lap: Max Verstappen (Red Bull) - 1 :17.988 ... Fastest Lap: George Russell (Mercedes) - 1 :14.119, title: F1 Schedule & Results - 2025 Formula 1 Season, link: https://www.skysports.com/f1/schedule-results
3. snippet: Home, title: One.com Web hosting - Domain •, link: https://www.one.com/
4. snippet: Ligue 1 : Nice enfonce Rennes dans la crise, Le Havre fait une superbe opération ... A quelle place va terminer Strasbourg en Ligue 1 ?, title: Ligue 1 2025/2026 : Résultats Actu But - Foot Mercato, link: https://www.footmercato.net/france/ligue-1/

----- WEB SEARCH -----
영화 박스오피스 순위
w
1. snippet: [박스오피스] 코너는 실시간 발권데이터를 전일기준까지 반영하여 일별/주간/주말/기간별

In [20]:
result

{'messages': [HumanMessage(content='현재 박스오피스 1위 영화는?', additional_kwargs={}, response_metadata={}, id='886a25d7-8602-47fa-ab70-754a5f031eca'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_web_search', 'arguments': '{"query": "\\ud604\\uc7ac \\ubc15\\uc2a4\\uc624\\ud53c\\uc2a4 1\\uc704 \\uc601\\ud654"}'}, '__gemini_function_call_thought_signatures__': {'ff72827f-3fbe-4c78-ade6-796a3e85588c': 'CpgEAdHtim90PGs5uoY62TpO9t3ZdB/gsH7LTRBvxmooqtoN2FzuZA82ho4YvOTAz0Sk6yvqi1Fjy/bYuRqs1GQvkABDmTWH6rmqY/TzOZBHyMj/MqzpyRS0CW1AwfWmUT7DSru+xrWFbKJTl7zdWE/89I+elf7G9NLjJaY1n66fO0z4gGs31I5xSLRKKUlhNyavvH/aJiCv6FBN9e+6If4cuiQaAlZK6ahUe2yRkHiXpjG0hH1+dLBDwy0HLOUwH8ByhXBU+LsDmZ7LUy5P3i7Twd6JgVtbONBZn1gPjnCzY5oWKY1E/di2MgHxxvgb7E/p8RAlNg42KkOW5bjLXIkFVqVM1M9uZKXuNxglS3b6IyUcxoCdOCtpKRdyRIyrbBFPCrmXupvQoPxDSnHYlcGyvWtzlHK7KVilGJod38zkgqr82OJ9l6AVvpZoYuATfXHFOeXN/h03Firt0rPP19SGKoCCYTGGmd5gY/D6hOwIA6yMqFROeYbokO03SuXjjjQbWeRCuwe5I9kK7IK+kSjiAMuPcsWqRkpfUJxj2GaJEJ0ylJEqmRuDK8njl6aw